# rlmflow visualization walkthrough

A guided tour of every visualization, query, and export utility shipped with rlmflow. Everything here runs **offline** against the saved trace under `examples/coding-agent/workspace/` — the boids simulation that the coding-agent example built. No API keys, no LLM calls.

**What we cover**

1. Loading a saved trace
2. Inline tree rendering (`state.tree()`, Jupyter auto-render, `ascii_boxes`)
3. Querying the typed graph (`find`, `leaves`, `where`, `errors`, `path_to`, `diff`)
4. Topology renders (mermaid · flowchart · sequence · dot · d2)
5. Step-indexed timeline (gantt)
6. Per-node detail (`code_log`, `error_summary`, `message_stream`, `diff_system_prompts`)
7. Cost & reports (`token_sparkline`, `budget_burndown`, `report_md`)
8. Comparison + streaming (`bench_table`, `tee`, `json_logs`)
9. CLI equivalents

Full roadmap: [`docs/internal/visualization_roadmap.md`](../../docs/internal/visualization_roadmap.md).

## 1. Load the trace

A trace is a JSON file holding the list of states the engine visited plus optional metadata. `load_trace` returns a `Trace` named tuple with `states` and `metadata`.

In [10]:
from rlmflow.utils.trace import load_trace

trace = load_trace("../data/notebook-coding-agent/trace")
states = trace.states
agents = {n.agent_id for s in states for n in s.walk()}
print(f"loaded {len(states)} states across {len(agents)} agents")
print(f"agents : {sorted(agents)}")
print(f"metadata: {trace.metadata or '(none)'}")

loaded 5 states across 7 agents
agents : ['root', 'root.boid.js', 'root.flock.js', 'root.index.html', 'root.main.js', 'root.styles.css', 'root.utils.js']
metadata: {'task': 'Create a simple boids simulation in plain html and javascript. Make them fast, colorful, and add trails. Do not add configurations -- just the canvas. Split each component into separate files and delegate the work. The main runnable interface should be in index.html. Save in output/boids-simulation.'}


## 2. Inline tree rendering

Every `Node` defines `_repr_html_` and `_repr_mimebundle_`, so just returning a node from a Jupyter cell renders it as a styled tree. Trees can collapse on the final `ResultNode`, so we pick the *richest* state for a complete picture.

In [11]:
richest = max(states, key=lambda s: len(s.walk()))
richest

root [supervising] {default:gpt-5}
  root.index.html [query] {fast}
  root.styles.css [query] {fast}
  root.main.js [query] {fast}
  root.flock.js [query] {fast}
  root.boid.js [query] {fast}
  root.utils.js [query] {fast}

In [12]:
print(richest.tree())

root [supervising] {default:gpt-5}
  root.index.html [query] {fast}
  root.styles.css [query] {fast}
  root.main.js [query] {fast}
  root.flock.js [query] {fast}
  root.boid.js [query] {fast}
  root.utils.js [query] {fast}


In [13]:
from rlmflow.utils.viz import ascii_boxes

print(ascii_boxes(richest))

╭──────────────────────────────────────── root  [supervising]  {default:gpt-5} ────────────────────────────────────────╮
│                                                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
├── ╭──────────────────────────────────────── root.index.html  [query]  {fast} ────────────────────────────────────────╮
│   │                                                                                                                  │
│   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
├── ╭──────────────────────────────────────── root.styles.css  [query]  {fast} ────────────────────────────────────────╮
│   │                                                                                                                  │
│   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
├── ╭───────────────────────────────────────── root.main.js  [query]  {fast} ──────────────────────────────────────────╮
│   │                                                                                                                  │
│   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
├── ╭───────────────────────────────────────── root.flock.js  [query]  {fast} ─────────────────────────────────────────╮
│   │                                                                                                                  │
│   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
├── ╭───────────────────────────────────────── root.boid.js  [query]  {fast} ──────────────────────────────────────────╮
│   │                                                                                                                  │
│   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
└── ╭───────────────────────────────────────── root.utils.js  [query]  {fast} ─────────────────────────────────────────╮
    │                                                                                                                  │
    ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── root  [supervising]  {default:gpt-5} ────────────────────────────────────────╮
│                                                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
├── ╭──────────────────────────────────────── root.index.html  [query]  {fast} ────────────────────────────────────────╮
│   │                                                                                                                  │
│   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
├── ╭──────────────────────────────────────── root.styles.css  [query]  {fast} ────────────────────────────────────────╮
│   │                                                                                                                  │
│   ╰───────────────────────────

## 3. Querying the typed graph

Every method below lives on `Node` — no I/O, no rendering, just graph queries.

In [14]:
print("all agents in subtree:")
for n in richest.walk():
    print(f"  {n.agent_id} [{n.type}]")

all agents in subtree:
  root [supervising]
  root.index.html [query]
  root.styles.css [query]
  root.main.js [query]
  root.flock.js [query]
  root.boid.js [query]
  root.utils.js [query]


In [15]:
print("leaves :", [n.agent_id for n in richest.leaves()])
print("results:", [n.agent_id for n in richest.results()])
print("errors :", richest.errors() or "(none)")

leaves : ['root.index.html', 'root.styles.css', 'root.main.js', 'root.flock.js', 'root.boid.js', 'root.utils.js']
results: []
errors : (none)


In [16]:
target = "root.boid.js"
hit = richest.find(target)
print(hit.agent_id, hit.type, "->", str(getattr(hit, "result", ""))[:60])

print(f"\npath_to({target!r}):")
for n in richest.path_to(target):
    print(f"  {n.agent_id} [{n.type}]")

root.boid.js query -> 

path_to('root.boid.js'):
  root [supervising]
  root.boid.js [query]


In [17]:
deep_results = richest.where(type="result", depth=1)
print([n.agent_id for n in deep_results])

deep_with_predicate = richest.where(lambda n: n.depth >= 1 and n.type == "result")
print([n.agent_id for n in deep_with_predicate])

[]
[]


In [18]:
# state.diff(other): which nodes appeared between two snapshots?
early = states[2]
late  = states[-2] if len(states) > 2 else states[-1]
diff  = late.diff(early)
print(f"{len(diff.added)} nodes added between step 2 and step {len(states) - 2}")
for n in diff.added[:6]:
    print(f"  + {n.agent_id} [{n.type}]")

1 nodes added between step 2 and step 3
  + root.flock.js [result]


## 4. Topology renders

Every renderer takes a node and returns text. Mermaid blocks render inline via `IPython.display.Markdown`. The same renders are available from the CLI as `rlmflow render <path> -f F`.

In [19]:
from IPython.display import Markdown
from rlmflow.utils.export import to_mermaid

Markdown(f"```mermaid\n{to_mermaid(richest)}\n```")

```mermaid
stateDiagram-v2
    state "root (supervising)" as n_79341770723a
    [*] --> n_79341770723a
    n_79341770723a --> n_be98c9fd9851
    state "root.index.html (query)" as n_be98c9fd9851
    n_79341770723a --> n_e95c272b90c6
    state "root.styles.css (query)" as n_e95c272b90c6
    n_79341770723a --> n_a6ffd1e972f3
    state "root.main.js (query)" as n_a6ffd1e972f3
    n_79341770723a --> n_3f11510eb976
    state "root.flock.js (query)" as n_3f11510eb976
    n_79341770723a --> n_8f501d291960
    state "root.boid.js (query)" as n_8f501d291960
    n_79341770723a --> n_3e96bed24f6a
    state "root.utils.js (query)" as n_3e96bed24f6a
```

In [20]:
from rlmflow.utils.export import to_mermaid_flowchart

Markdown(f"```mermaid\n{to_mermaid_flowchart(richest)}\n```")

```mermaid
flowchart TD
    n_79341770723a["root<br/><i>supervising</i>"]:::sup
    n_79341770723a --> n_be98c9fd9851
    n_be98c9fd9851["root.index.html<br/><i>query</i>"]:::query
    n_79341770723a --> n_e95c272b90c6
    n_e95c272b90c6["root.styles.css<br/><i>query</i>"]:::query
    n_79341770723a --> n_a6ffd1e972f3
    n_a6ffd1e972f3["root.main.js<br/><i>query</i>"]:::query
    n_79341770723a --> n_3f11510eb976
    n_3f11510eb976["root.flock.js<br/><i>query</i>"]:::query
    n_79341770723a --> n_8f501d291960
    n_8f501d291960["root.boid.js<br/><i>query</i>"]:::query
    n_79341770723a --> n_3e96bed24f6a
    n_3e96bed24f6a["root.utils.js<br/><i>query</i>"]:::query
    classDef query    fill:#1f6feb22,stroke:#58a6ff,color:#c9d1d9;
    classDef obs      fill:#1f6feb22,stroke:#58a6ff,color:#c9d1d9;
    classDef action   fill:#d2992222,stroke:#d29922,color:#c9d1d9;
    classDef sup      fill:#bc8cff22,stroke:#bc8cff,color:#c9d1d9;
    classDef resume   fill:#7ee78722,stroke:#7ee787,color:#c9d1d9;
    classDef err      fill:#f8514922,stroke:#f85149,color:#c9d1d9;
    classDef result   fill:#3fb95022,stroke:#3fb950,color:#c9d1d9;
```

In [21]:
from rlmflow.utils.export import to_mermaid_sequence

Markdown(f"```mermaid\n{to_mermaid_sequence(richest)}\n```")

```mermaid
sequenceDiagram
    participant root as root
    participant root_index_html as root.index.html
    participant root_styles_css as root.styles.css
    participant root_main_js as root.main.js
    participant root_flock_js as root.flock.js
    participant root_boid_js as root.boid.js
    participant root_utils_js as root.utils.js
    root->>+root_index_html: delegate
    root->>+root_styles_css: delegate
    root->>+root_main_js: delegate
    root->>+root_flock_js: delegate
    root->>+root_boid_js: delegate
    root->>+root_utils_js: delegate
```

In [22]:
from rlmflow.utils.export import to_dot, to_d2

print("--- DOT (paste into a Graphviz renderer) ---")
print(to_dot(richest))
print()
print("--- D2 (https://d2lang.com) ---")
print(to_d2(richest))

--- DOT (paste into a Graphviz renderer) ---
digraph rlmflow {
    rankdir=TB;
    node [shape=box, style="rounded,filled", fontname="Helvetica"];
    edge [fontname="Helvetica", fontsize=10];
    n_79341770723a [label="root\nsupervising", fillcolor="#bc8cff22", color="#bc8cff"];
    n_79341770723a -> n_be98c9fd9851;
    n_be98c9fd9851 [label="root.index.html\nquery", fillcolor="#58a6ff22", color="#58a6ff"];
    n_79341770723a -> n_e95c272b90c6;
    n_e95c272b90c6 [label="root.styles.css\nquery", fillcolor="#58a6ff22", color="#58a6ff"];
    n_79341770723a -> n_a6ffd1e972f3;
    n_a6ffd1e972f3 [label="root.main.js\nquery", fillcolor="#58a6ff22", color="#58a6ff"];
    n_79341770723a -> n_3f11510eb976;
    n_3f11510eb976 [label="root.flock.js\nquery", fillcolor="#58a6ff22", color="#58a6ff"];
    n_79341770723a -> n_8f501d291960;
    n_8f501d291960 [label="root.boid.js\nquery", fillcolor="#58a6ff22", color="#58a6ff"];
    n_79341770723a -> n_3e96bed24f6a;
    n_3e96bed24f6a [label="root.ut

## 5. Step-indexed timeline

`viz.gantt` prints a Rich table — one row per agent, one column per step, single-letter glyphs (`Q`/`A`/`O`/`S`/`R`/`F`/`E`). `gantt_html` writes a self-contained HTML page; we render it inline below.

In [23]:
from rlmflow.utils.viz import gantt

gantt(states)

┏━━━━━━━━━━━━━━━┳━┳━┳━┳━┳━┓
┃agent          ┃0┃1┃2┃3┃4┃
┡━━━━━━━━━━━━━━━╇━╇━╇━╇━╇━┩
│root           │Q│S│S│S│F│
│root.index.html│ │Q│F│F│F│
│root.styles.css│ │Q│F│F│F│
│root.main.js   │ │Q│F│F│F│
│root.flock.js  │ │Q│O│F│F│
│root.boid.js   │ │Q│F│F│F│
│root.utils.js  │ │Q│F│F│F│
└───────────────┴─┴─┴─┴─┴─┘

In [24]:
from IPython.display import HTML
from rlmflow.utils.viz import gantt_html

HTML(gantt_html(states, title="boids run"))

## 6. Per-node detail

Drill into the actual code that ran, errors that happened, and the chat-log view for any agent.

In [32]:
from rlmflow.utils.viz import code_log

print(code_log(states))

# [root] supervising
# Define the shared contract and delegate per-file work, then verify assembly.

requirements = ""
contract = '''
Project: output/boids-simulation

Overall requirements:
- Plain HTML + JavaScript (ES modules), no frameworks.
- No configuration UI — just a full-window canvas running the boids simulation.
- Boids should be fast, colorful, and leave trails.
- The main runnable interface is index.html.
- Split components into separate files as listed below.
- Use only the specified filenames and relative imports exactly as written.

File responsibilities (CONTRACT):
1) index.html
   - Minimal HTML document with:
     * <link rel="stylesheet" href="styles.css"> in <head>.
     * Exactly one canvas: <canvas id="boids"></canvas> inside <body>.
     * At the end of <body>, include: <script type="module" src="main.js"></script>
   - Title: "Boids Simulation"
   - No other UI elements.

2) styles.css
   - Make the canvas fill the entire window.
   - Black background, no scrol

In [33]:
# The boids run had no errors. Synthesize a small example so
# error_summary has something to group.
from rlmflow import ErrorNode, QueryNode
from rlmflow.utils.viz import error_summary

demo = QueryNode(
    agent_id="root",
    children=[
        ErrorNode(agent_id="root.a", error="no_code_block", content="missing repl block"),
        ErrorNode(agent_id="root.b", error="no_code_block", content="empty reply"),
        ErrorNode(agent_id="root.c", error="orphaned_delegates", content="never waited"),
    ],
)
print(error_summary(demo))

3 error(s) across 2 kind(s):
  no_code_block: 2
    └─ missing repl block
  orphaned_delegates: 1
    └─ never waited


In [37]:
from pathlib import Path
from rlmflow.workspace import FileSession
from rlmflow.utils.viz import message_stream

session = FileSession(Path("../data/notebook-coding-agent/session"))
stream = message_stream("root.index.html", session)
print(stream[:1800] + ("\n..." if len(stream) > 1800 else ""))

--- system ---
## Role

You are a recursive agent with a Python REPL. You solve tasks by writing and executing Python programs, and you can delegate subtasks to sub-agents with fresh context windows.

## REPL

- Every response MUST contain exactly one ```repl``` code block.
- Tools are already in the REPL namespace — call them directly.
- Variables persist across turns.
- `AGENT_ID`, `DEPTH`, `MAX_DEPTH` are set. You cannot delegate when `DEPTH == MAX_DEPTH`.
- Call `done(answer)` when finished.
- **Decompose programmatically** — when a task has independent parts, write code that creates those subtasks, delegates them, waits for results, and combines the answers.
- **Prefer parallel delegation for independent work** — build a list of handles and call `yield wait(*handles)` once instead of doing independent subtasks sequentially.
- **Explore before solving** — when files, long context, unknown data schemas, or tool outputs matter, first inspect samples, lengths, keys, or representative 

In [38]:
from rlmflow.utils.viz import diff_system_prompts

by_agent = {n.agent_id: n for n in session.load().values()}
diff = diff_system_prompts(by_agent["root.index.html"], by_agent["root.styles.css"])
print(diff[:1500] if diff != "(prompts identical)" else diff)

(prompts identical)


## 7. Cost & reports

Three views: a one-line ASCII sparkline of cumulative tokens, a budget burndown bar, and a full Markdown report bundling everything we've seen so far.

In [39]:
from rlmflow.utils.viz import token_sparkline, budget_burndown

print(token_sparkline(states))
print(budget_burndown(states))
print(budget_burndown(states, max_budget=50_000))

 ▁▆██   43708 tok over 5 steps
[████████████████████████████████████████] 100.0%  43708 tok (peak)
[███████████████████████████████████·····]  87.4%  43708/50000


In [40]:
from rlmflow.utils.viz import report_md

Markdown(report_md(states, title="boids — coding-agent run"))

# boids — coding-agent run

**Steps:** 5
**Agents:** 7
**Tokens:** 43,708 (28,684 in, 15,024 out)
**Outcome:** result

## Tree

```
root [result] {default:gpt-5} -> Boids simulation created in output/boids-simulation with split files and trails.
  root.index.html [result] {fast:gpt-5-mini} -> ok
  root.styles.css [result] {fast:gpt-5-mini} -> ok
  root.main.js [result] {fast:gpt-5-mini} -> ok
  root.flock.js [result] {fast:gpt-5-mini} -> ok
  root.boid.js [result] {fast:gpt-5-mini} -> ok
  root.utils.js [result] {fast:gpt-5-mini} -> ok
```

## Cumulative tokens

```
 ▁▆██   43708 tok over 5 steps
```

## Result

```
Boids simulation created in output/boids-simulation with split files and trails. Open index.html to run.
```


## 8. Comparison + streaming

`bench_table` aggregates labeled traces (use it to compare benchmark runs). `tee` fans a step iterator out to multiple sinks — handy for live runs. `json_logs` writes a newline-delimited event log for external tools (Loki / Datadog / DuckDB).

In [41]:
from rlmflow.utils.viz import bench_table

# Fake two runs by trimming, just to demonstrate the table shape.
print(bench_table({
    "boids:full":      states,
    "boids:firsthalf": states[: len(states) // 2 + 1],
}))

label            steps  agents  outcome  tokens  errors
---------------  -----  ------  -------  ------  ------
boids:full       5      7       result   43,708  0     
boids:firsthalf  3      7       open     33,766  0     


In [42]:
import tempfile
from rlmflow.utils.tracing import json_logs
from rlmflow.utils.viz import tee

events = []
for _ in tee(richest.walk(), events.append):
    pass
print(f"tee forwarded {len(events)} nodes to a sink")

with tempfile.NamedTemporaryFile(suffix=".jsonl", delete=False) as f:
    out = json_logs(states, f.name)
lines = out.read_text().splitlines()
print(f"json_logs wrote {len(lines)} lines to {out}")
print("first line preview:")
print(lines[0][:200] + "...")

tee forwarded 7 nodes to a sink
json_logs wrote 16 lines to /var/folders/lz/f4qx65kx5dn_5mxblzh2s5h00000gn/T/tmp0f65vfnw.jsonl
first line preview:
{"type": "query", "id": "n_034064b055f0", "branch_id": "main", "children": [], "agent_id": "root", "depth": 0, "query": "Create a simple boids simulation in plain html and javascript. Make them fast, ...


## 9. CLI equivalents

Most of the above is also available without writing Python. From a shell:

```bash
rlmflow view   examples/data/notebook-coding-agent/trace
rlmflow render examples/data/notebook-coding-agent/trace -f mermaid-flowchart
rlmflow render examples/data/notebook-coding-agent/trace -f gantt-html  -o gantt.html
rlmflow render examples/data/notebook-coding-agent/trace -f report-md   -o report.md
rlmflow render examples/data/notebook-coding-agent/trace -f code-log
rlmflow render examples/data/notebook-coding-agent/trace -f error-summary
rlmflow render examples/data/notebook-coding-agent/trace -f tokens
```

All formats: `mermaid` · `mermaid-flowchart` · `mermaid-sequence` · `dot` · `d2` · `tree` · `ascii-boxes` · `gantt-html` · `report-md` · `code-log` · `error-summary` · `tokens`.